In [1]:
%pip install openmeteo-requests requests-cache retry-requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

eia = pd.read_csv("../data/raw/eia_energy_data.csv", parse_dates=["datetime_utc"])
print(eia.shape)
eia.head()

(70088, 7)


,datetime_utc,region,demand_mwh,hour,day_of_week,month,is_weekend
0,2025-06-20 05:00:00,BPAT,6832,5,4,6,0
1,2025-06-20 06:00:00,BPAT,6542,6,4,6,0
2,2025-06-20 07:00:00,BPAT,6181,7,4,6,0
3,2025-06-20 08:00:00,BPAT,5915,8,4,6,0
4,2025-06-20 09:00:00,BPAT,5774,9,4,6,0


In [3]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Setup connection to Open-Meteo with caching and retry
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Region -> (city, latitude, longitude)
regions = {
    "BPAT": ("Seattle",      47.6062, -122.3321),
    "CISO": ("Sacramento",   38.5816, -121.4944),
    "ERCO": ("Dallas",       32.7767,  -96.7970),
    "ISNE": ("Boston",       42.3601,  -71.0589),
    "MISO": ("Chicago",      41.8781,  -87.6298),
    "NYIS": ("New York",     40.7128,  -74.0060),
    "PJM":  ("Philadelphia", 39.9526,  -75.1652),
    "SWPP": ("Oklahoma City",35.4676,  -97.5164),
}

START = "2025-06-20"
END   = "2026-06-20"

url = "https://archive-api.open-meteo.com/v1/archive"
all_weather = []

for region, (city, lat, lon) in regions.items():
    print(f"Fetching {city} for {region}...")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START,
        "end_date": END,
        "hourly": "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m",
        "temperature_unit": "fahrenheit"    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    hourly = response.Hourly()

    hourly_data = {
        "datetime_utc": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature":   hourly.Variables(0).ValuesAsNumpy(),
        "apparent_temp": hourly.Variables(1).ValuesAsNumpy(),
        "humidity":      hourly.Variables(2).ValuesAsNumpy(),
        "precipitation": hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed":    hourly.Variables(4).ValuesAsNumpy(),
    }
    hourly_data["region"] = region

    df = pd.DataFrame(data=hourly_data)
    all_weather.append(df)

weather_df = pd.concat(all_weather, ignore_index=True)
weather_df["datetime_utc"] = weather_df["datetime_utc"].dt.tz_localize(None)

print("\nDone! Weather data shape:", weather_df.shape)
weather_df.head()

Fetching Seattle for BPAT...
Fetching Sacramento for CISO...
Fetching Dallas for ERCO...
Fetching Boston for ISNE...
Fetching Chicago for MISO...
Fetching New York for NYIS...
Fetching Philadelphia for PJM...
Fetching Oklahoma City for SWPP...

Done! Weather data shape: (70272, 7)


,datetime_utc,temperature,apparent_temp,humidity,precipitation,wind_speed,region
0,2025-06-20 00:00:00,67.190002,60.940262,35.814621,0.0,14.050382,BPAT
1,2025-06-20 01:00:00,66.110001,60.363770,37.047642,0.0,12.065372,BPAT
2,2025-06-20 02:00:00,64.849998,58.896656,40.367939,0.0,13.684735,BPAT
3,2025-06-20 03:00:00,62.959999,58.055893,47.864166,0.0,11.901798,BPAT
4,2025-06-20 04:00:00,61.160000,57.362484,54.819580,0.0,9.361730,BPAT


In [4]:
weather_df = weather_df.rename(columns={
    "temperature":   "temperature_F",
    "apparent_temp": "apparent_temp_F",
    "humidity":      "humidity_pct",
    "precipitation": "precipitation_mm",
    "wind_speed":    "wind_speed_kmh",
})

weather_df.head()

,datetime_utc,temperature_F,apparent_temp_F,humidity_pct,precipitation_mm,wind_speed_kmh,region
0,2025-06-20 00:00:00,67.190002,60.940262,35.814621,0.0,14.050382,BPAT
1,2025-06-20 01:00:00,66.110001,60.363770,37.047642,0.0,12.065372,BPAT
2,2025-06-20 02:00:00,64.849998,58.896656,40.367939,0.0,13.684735,BPAT
3,2025-06-20 03:00:00,62.959999,58.055893,47.864166,0.0,11.901798,BPAT
4,2025-06-20 04:00:00,61.160000,57.362484,54.819580,0.0,9.361730,BPAT


In [5]:
merged = eia.merge(weather_df, on=["datetime_utc", "region"], how="left")

print("Merged shape:", merged.shape)
print("Missing weather values:", merged["temperature_C"].isna().sum())
merged.head()

Merged shape: (70088, 12)


KeyError: 'temperature_C'

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)

merged.to_csv("../data/processed/eia_with_weather.csv", index=False)
print("Saved!")

In [ ]:
weather_df.to_csv("../data/raw/weather_data.csv", index=False)
print("Saved!")